In [2]:
# Importando Bibliotecas 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsRegressor

print("Bibliotecas Importadas!")

Bibliotecas Importadas!


In [5]:
# Carregando os dados que tratei manualmente pelo excel
df = pd.read_csv('../data/processed/MICRODADOS_ENCCEJA_2024_REG_NAC.csv', 
                 sep=';', 
                 encoding='latin-1', 
                 )
print("Deu certo")

Deu certo


In [14]:
# Removendo valores nulos das notas 
colunas_notas = ['NU_NOTA_LC', 'NU_NOTA_CH', 'NU_NOTA_MT', 'NU_NOTA_CN', 'NU_NOTA_REDACAO']
df = df.dropna(subset=colunas_notas)

# Removendo valores nulos das colunas socioeconômicas 

colunas_perfil = ['Q11- Escolaridade Anterior', 'Q44 - Trabalha_ou_Nao', 'Q50- Faixa_Renda_Familiar']
df = df.dropna(subset=colunas_perfil)

# Renomeando as colunas 

df = df.rename(columns={
    'Q11- Escolaridade Anterior': 'q11_escolaridade_anterior',
    'Q44 - Trabalha_ou_Nao': 'q44_trabalha_ou_nao',
    'Q50- Faixa_Renda_Familiar': 'q50_faixa_renda_familiar'
})

# Descartando a coluna NU_INSCRICAO já que não tem utilidade no modelo.
df.drop(columns=['NU_INSCRICAO'], inplace=True, errors='ignore')

print("Limpeza concluída e colunas renomeadas!")
print(df.columns) 

Limpeza concluída e colunas renomeadas!
Index(['TP_CERTIFICACAO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'SG_UF_PROVA',
       'NU_NOTA_LC', 'NU_NOTA_CH', 'NU_NOTA_MT', 'NU_NOTA_CN',
       'IN_APROVADO_LC', 'IN_APROVADO_CH', 'IN_APROVADO_MT', 'IN_APROVADO_CN',
       'NU_NOTA_REDACAO', 'q11_escolaridade_anterior', 'q44_trabalha_ou_nao',
       'q50_faixa_renda_familiar'],
      dtype='object')


In [19]:
# ALGORITMO KNN
# Definindo quais colunas usaremos para prever (X) e quais queremos prever (y)
# Deixei TP_FAIXA_ETARIA fora do get_dummies pois ela já é uma escala numérica lógica
colunas_para_dummy = [
    'TP_SEXO', 
    'SG_UF_PROVA', 
    'q11_escolaridade_anterior', 
    'q44_trabalha_ou_nao', 
    'q50_faixa_renda_familiar',
    'TP_CERTIFICACAO'
]

colunas_notas = ['NU_NOTA_LC', 'NU_NOTA_CH', 'NU_NOTA_MT', 'NU_NOTA_CN', 'NU_NOTA_REDACAO']

# 2. Aplicando One-Hot Encoding (Transformando categorias em colunas de 0 e 1)
# O drop_first=True evita a redundância (ex: se não é Masculino, só pode ser Feminino)
X = pd.get_dummies(df.drop(columns=colunas_notas), columns=colunas_para_dummy, drop_first=True)
y = df[colunas_notas]

print(f"Total de colunas após o Encoding: {X.shape[1]}")

# Divisão em Treino e Teste (Holdout)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Implementação do KNN Regressor
# Usaremos 5 vizinhos para média de predição das notas
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Testando uma predição
# Pegando o primeiro aluno do conjunto de teste para ver o resultado
exemplo_aluno = X_test_scaled[0].reshape(1, -1)
predicao = knn.predict(exemplo_aluno)

print("\n--- TESTE DE PREDIÇÃO ---")
print(f"Notas esperadas para o perfil: \n{predicao}")

# Buscando os Vizinhos Mais Próximos 
distancias, indices = knn.kneighbors(exemplo_aluno)
vizinhos_reais = y_train.iloc[indices[0]]

print("\n--- COMPARATIVO COM VIZINHOS ---")
print("Média de notas dos 5 vizinhos mais parecidos:")
print(vizinhos_reais.mean())

Total de colunas após o Encoding: 52

--- TESTE DE PREDIÇÃO ---
Notas esperadas para o perfil: 
[[123.8  126.6  120.6  128.2    7.28]]

--- COMPARATIVO COM VIZINHOS ---
Média de notas dos 5 vizinhos mais parecidos:
NU_NOTA_LC         123.80
NU_NOTA_CH         126.60
NU_NOTA_MT         120.60
NU_NOTA_CN         128.20
NU_NOTA_REDACAO      7.28
dtype: float64
